## Configuration


In [ ]:
"""Train and evaluate FINDETECT XGBoost on Block D parquet buckets.

Requires a CUDA-enabled environment, project data at PROJECT_DIR, and enough
memory to load the validation and test periods.
"""

import glob
import multiprocessing
from pathlib import Path

import cudf
import duckdb
import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.metrics import average_precision_score, precision_score, recall_score, roc_auc_score

PROJECT_DIR = Path("/content/drive/MyDrive/AML_Project/aml-compliance-system-main")
BLOCK_D_PATH_GLOB = str(PROJECT_DIR / "Data/processed/FINDETECT_Block_D_Numerical_Final/block_d_numerical_bucket_*.parquet")
MODEL_PATH = PROJECT_DIR / "src/models/xgboost_full_period_model.json"
QUEUE_PATH = PROJECT_DIR / "Data/processed/High_Risk_Investigation_Queue_full_period.parquet"
TIMESTAMP_COL = "Timestamp"
TARGET_COL = "Is Laundering"

TRAIN_START, TRAIN_END = "2022-08-08 00:00:00", "2022-10-03 00:00:00"
VAL_START, VAL_END = "2022-10-03 00:00:00", "2022-10-17 00:00:00"
TEST_START, TEST_END = "2022-10-17 00:00:00", "2022-10-30 00:00:00"
assert TRAIN_END == VAL_START and VAL_END == TEST_START

FEATURE_COLS = [
    "Amount Received", "Amount Paid",
    "sender_out_degree", "sender_in_degree", "sender_unique_counterparties",
    "receiver_out_degree", "receiver_in_degree", "receiver_unique_counterparties",
    "sender_receiver_frequency", "bidirectional_relationship",
    "sender_outgoing_count_so_far_wk", "sender_unique_receivers_so_far_wk",
    "outgoing_count_24h", "outgoing_sum_24h", "outgoing_count_72h", "outgoing_sum_72h",
    "sender_total_sent_prev_week", "sender_avg_sent_prev_week",
    "sender_max_sent_prev_week", "sender_outgoing_count_prev_week",
    "receiver_total_received_prev_week", "receiver_avg_received_prev_week",
    "receiver_max_received_prev_week", "receiver_incoming_count_prev_week",
    "amount_vs_prev_max_diff", "amount_vs_prev_max_ratio",
    "transaction_frequency_diff", "transaction_frequency_ratio",
    "hours_since_last_incoming", "outgoing_to_incoming_ratio",
    "hour_of_day", "currency_mismatch", "same_bank", "payment_format_hash",
]

bucket_files = sorted(glob.glob(BLOCK_D_PATH_GLOB))
if not bucket_files:
    raise FileNotFoundError(f"No Block D parquet buckets found: {BLOCK_D_PATH_GLOB}")
print(f"Found {len(bucket_files)} Block D buckets")

con = duckdb.connect()
con.execute("PRAGMA temp_directory='/tmp/duckdb_temp'")
con.execute("PRAGMA preserve_insertion_order=false")
con.execute("PRAGMA memory_limit='75GB'")
con.execute(f"PRAGMA threads={multiprocessing.cpu_count()}")



## Data loading


In [ ]:


def build_query(file_path, start, end, include_accounts=False):
    """Read a half-open time window, with account IDs for test reporting if requested."""
    selected = [f'"{TIMESTAMP_COL}"', f'"{TARGET_COL}"']
    if include_accounts:
        selected.extend(['"Account"', '"Account_duplicated_0"'])

    for col in FEATURE_COLS:
        if col == "hour_of_day":
            selected.append("CAST(date_part('hour', \"Timestamp\") AS FLOAT) AS hour_of_day")
        elif col == "currency_mismatch":
            selected.append('CAST("Receiving Currency" != "Payment Currency" AS FLOAT) AS currency_mismatch')
        elif col == "same_bank":
            selected.append('CAST("From Bank" = "To Bank" AS FLOAT) AS same_bank')
        elif col == "payment_format_hash":
            selected.append('CAST(hash("Payment Format") AS FLOAT) AS payment_format_hash')
        else:
            selected.append(f'IF(isinf("{col}") OR isnan("{col}"), NULL, CAST("{col}" AS FLOAT)) AS "{col}"')

    columns = ", ".join(selected)
    return f"""
        SELECT {columns}
        FROM read_parquet('{file_path}')
        WHERE "{TIMESTAMP_COL}" >= TIMESTAMP '{start}'
          AND "{TIMESTAMP_COL}" < TIMESTAMP '{end}'
    """


class DuckDBPartitionIterator(xgb.DataIter):
    """Stream training buckets into XGBoost external-memory quantile batches."""

    def __init__(self, files, start, end, cache_dir):
        self.files = files
        self.start = start
        self.end = end
        self.position = 0
        self.positives = 0
        self.negatives = 0
        super().__init__(cache_prefix=cache_dir)

    def next(self, input_data):
        while self.position < len(self.files):
            file_path = self.files[self.position]
            self.position += 1
            table = con.execute(build_query(file_path, self.start, self.end)).arrow()
            if table.num_rows == 0:
                continue
            labels = table.column(TARGET_COL).to_numpy(zero_copy_only=False).astype(np.int32)
            self.positives += int((labels == 1).sum())
            self.negatives += int((labels == 0).sum())
            features = cudf.DataFrame.from_arrow(table.drop_columns([TIMESTAMP_COL, TARGET_COL]))
            input_data(data=features, label=cudf.Series(labels))
            return 1
        return 0

    def reset(self):
        self.position = 0
        self.positives = 0
        self.negatives = 0


def load_evaluation_window(start, end, include_accounts=False):
    table = con.execute(build_query(BLOCK_D_PATH_GLOB, start, end, include_accounts)).arrow()
    labels = table.column(TARGET_COL).to_numpy(zero_copy_only=False).astype(np.int32)
    if not len(labels):
        raise ValueError(f"No evaluation rows from {start} to {end}")
    accounts = None
    drop_columns = [TIMESTAMP_COL, TARGET_COL]
    if include_accounts:
        accounts = table.select(["Account", "Account_duplicated_0"]).to_pandas()
        drop_columns.extend(["Account", "Account_duplicated_0"])
    features = table.drop_columns(drop_columns).to_pandas()
    assert features.columns.tolist() == FEATURE_COLS
    return features, labels, accounts



## Model training


In [ ]:


X_val, y_val, _ = load_evaluation_window(VAL_START, VAL_END)
X_test, y_test, test_accounts = load_evaluation_window(TEST_START, TEST_END, include_accounts=True)
print(f"Validation: {len(y_val):,} transactions | Test: {len(y_test):,} transactions")

train_iter = DuckDBPartitionIterator(
    bucket_files, TRAIN_START, TRAIN_END, cache_dir="/content/xgb_cache_train"
)
dtrain = xgb.ExtMemQuantileDMatrix(train_iter, max_bin=256)
dval = xgb.QuantileDMatrix(X_val, label=y_val, ref=dtrain)
dtest = xgb.QuantileDMatrix(X_test, label=y_test, ref=dtrain)

params = {
    "tree_method": "hist",
    "device": "cuda",
    "objective": "binary:logistic",
    "eval_metric": "aucpr",
    "learning_rate": 0.05,
    "max_depth": 6,
    "scale_pos_weight": 30.0,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "max_bin": 256,
}
model = xgb.train(
    params,
    dtrain,
    num_boost_round=1500,
    evals=[(dval, "validation")],
    early_stopping_rounds=50,
    verbose_eval=50,
)
MODEL_PATH.parent.mkdir(parents=True, exist_ok=True)
model.save_model(str(MODEL_PATH))
print(f"Best iteration: {model.best_iteration}; model saved to {MODEL_PATH}")



## Evaluation and investigation queue


In [ ]:


val_probs = model.predict(dval)
test_probs = model.predict(dtest)
alert_budget = 0.005
threshold = float(np.percentile(val_probs, 100 * (1 - alert_budget)))
test_alerts = test_probs >= threshold

print(f"Validation threshold (top {alert_budget:.1%}): {threshold:.6f}")
print(f"Test AP: {average_precision_score(y_test, test_probs):.6f}")
print(f"Test ROC-AUC: {roc_auc_score(y_test, test_probs):.6f}")
print(f"Test alerts: {test_alerts.sum():,} / {len(test_alerts):,}")
print(f"Test precision: {precision_score(y_test, test_alerts, zero_division=0):.4f}")
print(f"Test recall: {recall_score(y_test, test_alerts, zero_division=0):.4f}")

assert len(test_accounts) == len(y_test) == len(test_probs)
scored = test_accounts.copy()
scored[TARGET_COL] = y_test  # Evaluation only; excluded from the exported queue.
scored["risk_probability"] = test_probs
sender = scored[["Account", TARGET_COL, "risk_probability"]].rename(columns={"Account": "account_id"})
receiver = scored[["Account_duplicated_0", TARGET_COL, "risk_probability"]].rename(
    columns={"Account_duplicated_0": "account_id"}
)
account_scores = (
    pd.concat([sender, receiver], ignore_index=True)
    .groupby("account_id", as_index=False)
    .agg(labeled_suspicious=(TARGET_COL, "max"), risk_score=("risk_probability", "max"))
)
queue_size = min(1000, len(account_scores))
ranked_accounts = account_scores.sort_values("risk_score", ascending=False)
queue_eval = ranked_accounts.head(queue_size)
print(f"Top {queue_size:,} accounts precision: {queue_eval['labeled_suspicious'].mean():.4f}")
print(f"Top {queue_size:,} accounts recall: {queue_eval['labeled_suspicious'].sum() / max(account_scores['labeled_suspicious'].sum(), 1):.4f}")

priority_queue = queue_eval[["account_id", "risk_score"]].copy()
QUEUE_PATH.parent.mkdir(parents=True, exist_ok=True)
priority_queue.to_parquet(QUEUE_PATH, index=False)
print(f"Investigation queue saved to {QUEUE_PATH}")
